In [70]:
from matplotlib import pyplot as plt
from serial import SerialException
import enum
import numpy as np

from discovery import Discovery
from driver import Driver
from load_pickle import load

try:
    Encryption_Board = Driver()
except SerialException:
    print("Encryption board not found. Please check the connection.")

try:
    Oscilloscope = Discovery()
except SystemExit:
    print("Oscilloscope not found. Please check the connection.")

# Setup reading channels for the oscilloscope
class CHANNELS(enum.Enum):
    POWER_CONSUMPTION = 0
    SBOX_ACCESS_TRIGGER = 1
    ENCRYPTION_START_TRIGGER = 2
    CH3 = 3
    CH4 = 4

Encryption board not found. Please check the connection.
-- ERROR: can not open device. Exiting...
Oscilloscope not found. Please check the connection.


In [71]:
def generateRandomPlaintext() -> int:
    return np.random.randint(0, 2**128)

def plotCurrentWaveform(measure: np.ndarray):
    plt.plot(measure)
    plt.xlabel("Time (ms)")
    plt.ylabel("Current (mA)")
    plt.title("Current waveform")
    plt.show()

## Board Functions

In [72]:
def encryptPlaintext(plaintext: int) -> int:
    ciphertext = Encryption_Board.encrypt(plaintext)
    return ciphertext

## Oscilloscope functions

In [73]:
def measureCurrent(measure: np.ndarray, rise: int, fall: int) -> float:
    """
    Digital pin 5:
        Rise: begin of 1st stage sbox calculation
        Fall: end of 1st stage sbox calculation
    """
    return measure[rise:fall].avg()

def prepareOscilloscope(channel: CHANNELS = CHANNELS.SBOX_ACCESS_TRIGGER):
    """
    Activates the specified channel as trigger to the readings
    if no <channel> is specified, the default is SBOX_ACCESS_TRIGGER
    """
    if channel is CHANNELS.SBOX_ACCESS_TRIGGER:
        Oscilloscope.configure() # Activate channel 0 and 1
    else:
        Oscilloscope.enableChannel(0)
        Oscilloscope.setMode('normal')
        Oscilloscope.enableChannel(channel.value)
        Oscilloscope.setOffset(channel.value, 0.0)
        Oscilloscope.setTriggerEdge(channel.value, 1, "rising")
    
    # Start the oscilloscope to be ready for readings
    Oscilloscope.run()

def waitForAcquisitionDone():
    while not Oscilloscope.isReady():
        pass

def getOscilloscopeReadings(channel: CHANNELS=CHANNELS.POWER_CONSUMPTION) -> np.ndarray:
    return np.array(
        Oscilloscope.getSamples(channel.value)
    )

def measurePowerTraces(num: int) -> tuple[list[int], np.ndarray]:
    T: list[int] = []
    M: np.ndarray = np.zeros((num, 4000)) # 4000 samples per trace (this number was taken from the provided traces)
    for i in range(num):
        plaintext = generateRandomPlaintext()
        T.append(plaintext)

        prepareOscilloscope(CHANNELS.SBOX_ACCESS_TRIGGER)
        ciphertext = Encryption_Board.encrypt(plaintext)
        waitForAcquisitionDone()
        M[i] = getOscilloscopeReadings(CHANNELS.POWER_CONSUMPTION)
    Oscilloscope.close()
    return T, M

# CPA Attack

In [74]:
def hamming_distance(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """
    Calculate the Hamming distance between two arrays of binary strings.
    A XOR B, then convert to bits and sum the number of 1s in each row
    """
    xor = np.bitwise_xor(a, b)
    bits_representation = np.unpackbits(xor, axis=1)
    hamming_distance = bits_representation.sum(axis=1)
    return hamming_distance.reshape(-1, 1) # Convert to column vector

# Attack using provided powertraces

In [75]:
P: list[int]
M: np.ndarray
P, M = load("./testsca.pickle")

In [76]:
print(len(P), M.shape)
print(P[0])

500 (500, 4000)
8178686806597379442190969825167098279


## CPA Implementation

In [81]:
from aes_sbox import SBOX

# Precompute helpers for the leakage model (HW of S-box output).
SBOX_ARRAY = np.array(SBOX, dtype=np.uint8)
HW_TABLE = np.array([bin(x).count("1") for x in range(256)], dtype=np.uint8)

def extract_plaintext_bytes(plaintexts: list[int]) -> np.ndarray:
    pt_bytes = np.zeros((len(plaintexts), 16), dtype=np.uint8)
    for i, p in enumerate(plaintexts):
        for b in range(16):
            shift = 8 * (15 - b)
            pt_bytes[i, b] = (p >> shift) & 0xFF
    return pt_bytes

def cpa_byte(pt_bytes: np.ndarray, traces: np.ndarray, byte_index: int) -> dict[int, float]:
    N, _ = traces.shape
    plaintexts_byte = pt_bytes[:, byte_index]

    # Yeah, list is enough as its a number (instead of dict),
    # but this is more intuitive
    correlations: dict[int, float] = {} 
    for key_guess in range(256):
        key_guess_bytes = np.uint8(key_guess)
        R = np.repeat(key_guess_bytes, N).reshape(-1, 1)
        H = hamming_distance(
            plaintexts_byte, R
        )
        numerator = N * np.sum(traces * H) - np.sum(traces) * np.sum(H)

        first_term = (N * np.sum(traces * traces) - np.sum(traces) ** 2)
        second_term = (N * np.sum(H * H) - np.sum(H) ** 2)
        if first_term <= 0 or second_term <= 0:
            correlation = 0 # Handle case where sqrt is negative or zero
        else: 
            denominator = np.sqrt(first_term) * np.sqrt(second_term)
            correlation = numerator / denominator if denominator != 0 else 0
        correlations[key_guess] = correlation
    return correlations

def cpa_attack(plaintexts: list[int], traces: np.ndarray) -> tuple[list[int], list[float]]:
    pt_bytes = extract_plaintext_bytes(plaintexts)
    key_bytes: list[int] = []
    key_scores: list[float] = []
    for b in range(16):
        correlations = cpa_byte(pt_bytes, traces, b)
        guess = max(correlations, key=lambda key_guess: correlations[key_guess])
        score = correlations[guess]
        key_bytes.append(guess)
        key_scores.append(score)
    return key_bytes, key_scores

key_bytes, key_scores = cpa_attack(P, M)
key_hex = "".join(f"{b:02x}" for b in key_bytes)
print(f"Recovered key: {key_hex}")

Recovered key: 00000000000000000000000000000000


In [82]:
import aes_encrypt
aes_encrypt.encrypt(0x0, int(key_hex, 16)).to_bytes(16, 'big').hex()

'2e2b34ca59fa4c883b2c8aefd44be966'